# Simulate a sky background with realistic background galaxies and Milky Way stars

This notebook shows how to generate a sky background model including background galaxies and MW stars. You are encouraged to use the script at `rosesim/scripts/sim_sky.py`. 

In [9]:
# Import packages
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS

import os, sys
# sys.path.append('/home/jiaxuanl/Research/SALAD/script/')
sys.path.append('/home/jiaxuanl/Research/Packages/romanisim/')
os.chdir('/scratch/gpfs/JENNYG/jiaxuanl/Data/SBF/Rosesim/')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import asdf
from astroquery.gaia import Gaia
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.table import Table, vstack, Column, hstack
from astropy import units as u
from astropy.visualization import simple_norm
import copy
import galsim
import roman_datamodels as rdm
from romanisim import gaia, bandpass, catalog, log, wcs, persistence, parameters, ris_make_utils as ris
from romanisim.image import inject_sources_into_l2
from romanisim.l3 import inject_sources_into_l3

## 1. Load JAGUAR background galaxies. 

JAGUAR is a simulated galaxy catalog, designed for JWST work. See here for more details: https://fenrir.as.arizona.edu/jaguar/download_jaguar_files.html

The catalog needs to be requested from that website. You can email me if you really want to use this catalog. 

In [11]:
# cat_Q = Table.read("/scratch/gpfs/JENNYG/jiaxuanl/Data/SBF/JAGUAR/JADES_Q_mock_r1_v1.2.fits")
# cat_SF = Table.read("/scratch/gpfs/JENNYG/jiaxuanl/Data/SBF/JAGUAR/JADES_SF_mock_r1_v1.2.fits")
# cat_all = vstack([cat_Q, cat_SF])
# cat_all = cat_all[(cat_all['sersic_n'] < 6) & (cat_all['sersic_n'] > 0.35)]
# cat_all.write('/scratch/gpfs/JENNYG/jiaxuanl/Data/SBF/JAGUAR/JADES_all_mock_r1_v1.2.fits')

## 2. Artpop and make catalogs for Romanisim

In [12]:
from kuaizi.display import display_single

In [13]:
import os
os.environ['NUMEXPR_MAX_THREADS'] = "30"
os.environ['ROSESIM_DATA_PATH'] = "/scratch/gpfs/JENNYG/jiaxuanl/Data/SBF/Rosesim/"

import artpop
import rosesim
from rosesim.rose import RomanGalaxy, RomanSky
rng = np.random.RandomState(100)

In [14]:
# This is chosen randomly. The MW stars from TRILEGAL are actually around this location.
obs_ra = 150.1049 #270.95
obs_dec = 2.2741 # -0.2

In [17]:
s = 5001 # size of the frame
sky = RomanSky(obs_ra, obs_dec, xy_dim=np.array([s, s]), prefix='sky_jaguar_test')
radius = sky.xy_dim[0] * 0.11 / 3600
sky.obs_time = Time('2025-01-01T00:00:00')
# sky.load_gaia_star(radius=radius)
sky.load_trilegal_star(radius=radius, area=0.05)
sky.load_jaguar_bkg(radius=radius, seed=42)
sky.gen_catalog(include_bkg=False, include_star=False)

Warning -- need to figure out the coefficients between JWST and Roman filters.
No sources to be included, so it will generate an empty sky.


In [ ]:
exptime = 642 # HLWAS
sky.observe('F158', exptime=exptime)
# sky.observe('F106', exptime=exptime)
# sky.observe('F129', exptime=exptime)

In [57]:
dm = rosesim.read_L3_asdf('./F158_642s.asdf')
rosesim.utils.asdf_to_fits(dm, 'F158_642s.fits', subtract_bkg=False)

Successfully wrote FITS file: F158_642s.fits


## 3. Visualize the sky background

In [18]:
from astropy.visualization import make_lupton_rgb

In [19]:
zp = -2.5 * np.log10(1 * ((0.11 * u.arcsec)**2).to(u.steradian).value * 1e6 / 3631)
print(zp)

25.265227862373404


In [20]:
os.chdir('../sky_jaguar_trilegal/')
os.chdir('../empty_sky/')

In [141]:
filters = ['F106', 'F129', 'F158']
imgs = []
exptime = 642
# exptime = 2568
# exptime = 2568 * 4
for filt in filters:
    af = asdf.open(f'./{filt}_{exptime}s.asdf')
    dm = rdm.open(af)
    img = dm.data - np.median(dm.data)
    img *= 10**(0.4 * (27 - zp))
    imgs.append(img[:, :])

In [142]:
# rgb = make_lupton_rgb(1.0 * imgs[2], 1.05 * imgs[1], 1.2 * imgs[0], minimum=-0.03, stretch=1, Q=4)
rgb = make_lupton_rgb(0.9 * imgs[2], 1.05 * imgs[1], 1.2 * imgs[0], minimum=0, stretch=1, Q=3)
fig, ax = plt.subplots(figsize=(20, 20))
plt.imshow(rgb)
plt.axis('off')

plt.savefig(f'/tigress/jiaxuanl/public_html/figure/SBF/Rosesim/sky_jaguar_trilegal_{exptime}s.png', bbox_inches='tight', dpi=200, transparent=True)
plt.close()

/scratch/gpfs/LSST/stacks/stack_20250106/conda/envs/lsst-scipipe-12.1.0/lib/python3.13/site-packages/astropy/visualization/lupton_rgb.py:645: RuntimeWarning: invalid value encountered in divide
  fInorm = np.where(Int <= 0, 0, np.true_divide(fI, Int))


Check https://tigress-web.princeton.edu/~jiaxuanl/figure/SBF/Rosesim/sky.png

---

In [143]:
dm = rosesim.read_L3_asdf("/scratch/gpfs/JENNYG/jiaxuanl/Data/SBF/Rosesim/sky_jaguar/F158_642s.asdf")

In [144]:
dm.meta

{'association': {'name': '?'}, 'cal_logs': <roman_datamodels.stnode.CalLogs object at 0x7f3ad447dae0>, 'cal_step': {'flux': 'COMPLETE', 'outlier_detection': 'COMPLETE', 'resample': 'COMPLETE', 'skymatch': 'COMPLETE'}, 'calibration_software_name': 'RomanCAL', 'calibration_software_version': '?', 'coadd_info': {'exposure_time': 642.0, 'individual_image_meta': None, 'max_exposure_time': 642.0, 'time_first': <Time object: scale='utc' format='mjd' value=60675.99628472222>, 'time_last': <Time object: scale='utc' format='mjd' value=60676.00371527778>, 'time_mean': <Time object: scale='utc' format='isot' value=2025-01-01T00:00:00.000>}, 'coordinates': {'reference_frame': 'ICRS'}, 'file_date': <FileDate object: scale='utc' format='isot' value=2020-01-01T00:00:00.000>, 'filename': 'F158_642s.asdf', 'individual_image_meta': {}, 'instrument': {'name': 'WFI', 'optical_element': 'F158'}, 'model_type': 'MosaicModel', 'observation': {'execution_plan': -999999, 'exposure': -999999, 'exposure_grouping':

In [22]:
gwcs = dm.meta.wcs

In [25]:
gwcs.footprint()

array([[150.02843971,   2.19769392],
       [150.02843161,   2.35050203],
       [150.18136839,   2.35050203],
       [150.18136029,   2.19769392]])

In [84]:
obs_ra / 15, obs_dec

(10.006993333333332, 2.2741)

'/scratch/gpfs/JENNYG/jiaxuanl/Data/SBF/Rosesim/sky_jaguar_trilegal'